# Step 7: QLoRA fine-tuning, Run A (rank 16, alpha 32, 1 epoch) -- Exp 4

Fine-tunes Qwen2.5-Coder-1.5B-Instruct with QLoRA (4-bit nf4) on the Step 6 training set
(`sft_train.jsonl` / `sft_val.jsonl`, prompt v2, no few-shot), loss on the SQL answer
tokens only.

**Before Run All:**
- Settings -> Accelerator: **GPU T4 x2** (only one T4 used, intentional), Internet: **On**.
- Add Input -> your uploaded `sft-training-data` dataset (zip of `sft_train.jsonl` +
  `sft_val.jsonl` from `data/processed/`).
- Add-ons -> Secrets: **`HF_TOKEN`** (write access, for pushing checkpoints/the final
  adapter to a private repo) and **`WANDB_API_KEY`**.
- Edit the single line marked `CHANGE ME` in the config cell below (your HF repo id)
  before running. **If you run this notebook again for a different config (Run B, a
  different rank, etc.), change `HF_REPO_ID` to a new repo** (e.g. `...-run-b`) --
  reusing Run A's repo would auto-resume from Run A's checkpoint instead of starting
  that config fresh.

**What this notebook does:**
- Tries **Unsloth** first; if installing it would change the on-disk torch/CUDA build
  (checked via a fresh subprocess, not the already-imported module -- see the vLLM
  lesson), it aborts Unsloth cleanly and falls back to **TRL-style PEFT + bitsandbytes**
  QLoRA instead. Either way, loss masking, data loading, training loop, checkpointing
  and logging are identical.
- LoRA on every attention and MLP projection (`q/k/v/o_proj`, `gate/up/down_proj`), fp16
  (T4 has no bf16), gradient checkpointing, `train_sampling_strategy="group_by_length"`
  to cut padding waste.
- **Per-device batch 1, gradient accumulation 32** (effective batch 32): with
  group-by-length sampling, a batch can contain several of the longest (~4,096-token)
  sequences together, which is the real memory risk on a T4 -- never batching more than one such
  sequence at once removes it. If a CUDA OOM still happens, the training loop saves an
  emergency checkpoint, halves the per-device batch (doubling accumulation to keep the
  effective batch size at 32), logs the change, and resumes automatically.
- Loss is computed only on the SQL answer tokens: `sft_train.jsonl` already stores
  `prompt_text` (identical structure to the Step 4+5 eval prompts) and `full_text`
  (prompt + target SQL). Both are re-tokenized here with this notebook's own tokenizer,
  and the token-count difference is how many leading tokens get masked out of the loss.
  Checked two ways before training starts: every example's tokenized prompt must be an
  exact prefix of its tokenized full text (stops with a list of mismatches if not), and
  one real example is printed showing the mask/train boundary and confirming `<|im_end|>`
  is in the trained region, not masked out.
- Validation uses a **fixed random (seed 42) 200-example subset** of `sft_val.jsonl`,
  drawn across all of its validation `db_id`s, not the first 200 rows (which Spider's
  json groups by `db_id`, so that would have skewed toward whichever one or two
  databases happen to come first in file order).
- A ~30-step smoke test measures real tokens/sec and projects the full epoch's time,
  then training continues automatically in the same run.
- Checkpoints (adapter + optimizer + scheduler + trainer state) push to a **private** HF
  Hub repo (visibility checked and printed explicitly, not just assumed from the create
  call) at `last-checkpoint` at least every ~25 minutes (and at the end). A fresh session
  auto-resumes from there if it exists.
- Logs to Weights & Biases, resuming the same run id across Kaggle sessions.
- At the end: final adapter pushed to the HF repo, loss-curve plot and run metadata
  saved to `/kaggle/working/`.

## Resuming an interrupted run
Checkpointing here goes through the HF Hub, not Kaggle's Notebook Output Files, so
resuming does **not** need the Add Input dance Step 4+5 used:
1. Just open the notebook again (new Kaggle session is fine) and **Run All**.
2. The notebook checks the HF repo for a `last-checkpoint` folder; if one exists, it
   downloads it and continues from that exact step, printing `RESUMING` and which step.
   Otherwise it prints `STARTING FRESH`.
3. The W&B run id is stored alongside the checkpoint, so resumed training logs into the
   same W&B run instead of starting a new one.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import json as _json
import os
import subprocess
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("transformers:", transformers.__version__)

# Packages a pip install must never be allowed to touch -- this is exactly what broke the
# Step 4+5 notebook's first vLLM attempt (pip silently upgraded torchaudio to a build that
# didn't match the already-loaded torch).
WATCHED_PACKAGES = ["torch", "torchvision", "torchaudio", "triton", "transformers"]


def installed_torch_build():
    """What's actually on disk right now, read via a FRESH subprocess -- not the
    already-imported `torch` module in this kernel, which can't reflect a pip install
    that happened after it was first imported."""
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
        capture_output=True, text=True,
    )
    lines = r.stdout.strip().splitlines()
    return (lines[0], lines[1]) if len(lines) >= 2 else (None, None)


TORCH_BUILD_AT_START = installed_torch_build()
print("torch build on disk, recorded for later comparison:", TORCH_BUILD_AT_START)


def _current_versions(names):
    out = {}
    for name in names:
        r = subprocess.run(
            [sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{name}'))"],
            capture_output=True, text=True,
        )
        out[name] = r.stdout.strip() if r.returncode == 0 else None
    return out


def dry_run_would_change_watched(args):
    """Simulate the install with `pip install --dry-run --report <file>` -- installs
    nothing -- and check whether the packages it WOULD install/upgrade include any of
    WATCHED_PACKAGES at a version different from what's currently on disk. This is the
    PRIMARY safety net: it must catch a bad install before anything is actually run, not
    just after. Returns (would_change: bool, details: dict)."""
    # OUTPUT_DIR is defined in the config cell, which runs before this function is ever
    # called (model-loading cell) -- not before this cell defines the function itself.
    report_path = os.path.join(OUTPUT_DIR, "_pip_dry_run_report.json")
    if os.path.exists(report_path):
        os.remove(report_path)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", "--quiet",
         "--report", report_path] + args,
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not os.path.exists(report_path):
        # can't simulate (old pip, or a real resolution error) -- treat as unsafe
        return True, {"error": (result.stderr or "no dry-run report produced").strip()[:500]}

    with open(report_path, encoding="utf-8") as f:
        report = _json.load(f)

    current = _current_versions(WATCHED_PACKAGES)
    changes = {}
    for item in report.get("install", []):
        meta = item.get("metadata", {})
        pkg_name = meta.get("name", "")
        pkg_version = meta.get("version")
        for watched in WATCHED_PACKAGES:
            if pkg_name.lower() == watched.lower() and pkg_version != current.get(watched):
                changes[watched] = {"current": current.get(watched), "would_become": pkg_version}
    return (len(changes) > 0), changes


def safe_pip_install(args, label):
    """Install `args`, but only after a dry run confirms it wouldn't touch any
    WATCHED_PACKAGES -- never install first and clean up after, since an install that
    upgrades torch's dependencies can break the already-loaded torch in this kernel
    before we'd get a chance to uninstall anything. Returns True if installed (or
    already satisfied), False if skipped because the dry run flagged a protected
    package. The post-install on-disk check is a second safety net only -- if IT ever
    trips, that means the dry run missed something, and we stop the notebook rather
    than continue on a possibly broken environment."""
    would_change, details = dry_run_would_change_watched(args)
    if would_change:
        print(f"{label}: SKIPPING install -- a dry run shows it would change a protected "
              f"package: {details}. Nothing was installed.")
        return False

    before = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)
    after = installed_torch_build()
    if after != before:
        raise RuntimeError(
            f"{label}: installing it changed the on-disk torch build from {before} to "
            f"{after} even though the dry-run check passed. Stopping the notebook rather "
            f"than continue -- the dry run isn't foolproof (e.g. a package's own post-install "
            f"script can still change things it doesn't declare as a dependency)."
        )
    return True


In [ ]:
# ---- Config -- edit HF_REPO_ID before running ----
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEED = 0

HF_REPO_ID = "CHANGE_ME/text2sql-qlora-run-a"  # <-- CHANGE ME: your HF username/repo
# IMPORTANT: Run B (or any other config) must use a DIFFERENT HF_REPO_ID than Run A
# (e.g. "...-run-b", not "...-run-a") -- this notebook auto-resumes from whatever
# last-checkpoint it finds at HF_REPO_ID, so reusing Run A's repo for a different
# config would silently resume training from Run A's weights under a new hyperparameter
# set, not start that config fresh.
WANDB_PROJECT = "text2sql-qlora"
WANDB_RUN_NAME = "run-a-rank16-alpha32-1epoch"

# LoRA (Run A)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# training set was built with this cutoff (Step 6); baseball_1 is the only db it drops
MAX_SEQ_LEN = 4096

# optimizer / schedule -- see the markdown note right after the Trainer is built for why
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.03
# per_device_batch=1 is deliberately conservative: with group_by_length, a batch can
# contain several of the longest (~4,096-token) sequences at once, and this model, even
# in 4-bit, is wide enough that attention memory at that length is the real risk on a
# T4's 16GB -- never batching more than one such sequence together removes that risk
# entirely. grad_accum makes up the rest of the effective batch size (unchanged at 32).
# train_with_oom_retry() below still halves further (and doubles accum) on an actual
# CUDA OOM, as a safety net, not the primary defense.
PER_DEVICE_BATCH = 1
GRAD_ACCUM_STEPS = 32         # effective batch size = 32
NUM_EPOCHS = 1                # Run A

SMOKE_TEST_STEPS = 30
TARGET_CHECKPOINT_INTERVAL_SEC = 25 * 60   # under the ~30 min requirement, on purpose
FIXED_VAL_SUBSET_SIZE = 200                 # random (seed 42) subset of sft_val.jsonl, across all val db_ids

OUTPUT_DIR = "/kaggle/working/train_out"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Fail fast on any TrainingArguments/Trainer argument mismatch, in seconds, before
# loading the ~3-4GB model. transformers versions have renamed or removed training
# arguments before (e.g. warmup_ratio and group_by_length no longer exist in some
# releases) -- this cell builds a real TrainingArguments and a real Trainer with the
# actual values we intend to train with, against a tiny dummy model and dataset, so any
# such mismatch fails here instead of after a multi-minute model download.
import math

import torch.nn as nn
import transformers
from transformers import Trainer, TrainerCallback, TrainingArguments

print("transformers version:", transformers.__version__)

_dummy_total_steps = 100  # placeholder; only the arg names/types are being checked here
_dummy_warmup_steps = math.ceil(WARMUP_RATIO * _dummy_total_steps)


def _build_args_for_check(per_device_batch, grad_accum, warmup_steps, eval_steps):
    return TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, "_arg_check"),
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        warmup_steps=warmup_steps,
        fp16=True,
        gradient_checkpointing=True,
        train_sampling_strategy="group_by_length",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="no",
        report_to=["wandb"],
        seed=SEED,
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
    )


_check_args = _build_args_for_check(
    PER_DEVICE_BATCH, GRAD_ACCUM_STEPS, _dummy_warmup_steps, max(1, _dummy_total_steps // 4)
)


class _TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 4)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.linear(input_ids.float())
        loss = out.sum() if labels is not None else None
        return {"loss": loss, "logits": out} if loss is not None else {"logits": out}


class _TinyCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        pass

    def on_step_end(self, args, state, control, **kwargs):
        pass

    def on_train_end(self, args, state, control, **kwargs):
        pass


_tiny_dataset = [{"input_ids": [1, 2, 3], "labels": [1, 2, 3]} for _ in range(4)]
_tiny_trainer = Trainer(model=_TinyModel(), args=_check_args, train_dataset=_tiny_dataset,
                         eval_dataset=_tiny_dataset, data_collator=lambda x: x)
_tiny_trainer.add_callback(_TinyCallback())
print("TrainingArguments + Trainer + TrainerCallback signatures all OK for this "
      "transformers version -- safe to proceed to model loading.")
del _check_args, _tiny_trainer


In [ ]:
# Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add-ons -> Secrets -> add HF_TOKEN "
                        "(a Hugging Face token with write access), then Run All again.")
try:
    WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
except Exception:
    raise RuntimeError("Missing Kaggle secret WANDB_API_KEY. Add-ons -> Secrets -> add "
                        "WANDB_API_KEY, then Run All again.")
print("secrets loaded OK")


In [ ]:
# Find the Step 6 training files under /kaggle/input (recursive search, never a fixed
# dataset slug -- see the Kaggle resumability rule in CLAUDE.md).
import glob


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None


missing = []
paths = {}
for fname in ("sft_train.jsonl", "sft_val.jsonl"):
    p = find_input_file(fname)
    if p is None:
        missing.append(fname)
    else:
        paths[fname] = p
        print(f"found {fname} -> {p}")

if missing:
    raise FileNotFoundError(
        "Missing required input file(s) under /kaggle/input: " + ", ".join(missing) +
        ". Make sure the dataset with sft_train.jsonl and sft_val.jsonl is attached."
    )

TRAIN_PATH = paths["sft_train.jsonl"]
VAL_PATH = paths["sft_val.jsonl"]


In [ ]:
# Make sure the private HF repo exists, then check it for a checkpoint to resume from.
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

# create_repo(exist_ok=True) does NOT change visibility if the repo already existed
# (e.g. created manually, or by an earlier run, as public) -- check explicitly and force
# it private rather than assuming the private=True above took effect.
repo_info = api.repo_info(repo_id=HF_REPO_ID)
if not repo_info.private:
    print(f"{HF_REPO_ID} exists but is NOT private -- forcing it private now.")
    api.update_repo_visibility(repo_id=HF_REPO_ID, private=True)
    repo_info = api.repo_info(repo_id=HF_REPO_ID)
print(f"HF repo {HF_REPO_ID} visibility: {'private' if repo_info.private else 'PUBLIC'}")
if not repo_info.private:
    raise RuntimeError(f"{HF_REPO_ID} is still public after trying to force it private -- "
                        f"stopping rather than push checkpoints to a public repo.")

RESUME_CKPT_DIR = None
RESUMED = False
try:
    repo_files = api.list_repo_files(repo_id=HF_REPO_ID)
    if any(f.startswith("last-checkpoint/") for f in repo_files):
        local_dir = os.path.join(OUTPUT_DIR, "resumed")
        snapshot_download(repo_id=HF_REPO_ID, local_dir=local_dir, allow_patterns="last-checkpoint/*")
        RESUME_CKPT_DIR = os.path.join(local_dir, "last-checkpoint")
        RESUMED = True
except Exception as e:
    print(f"could not check/download a previous checkpoint ({e}) -- starting fresh")

if RESUMED:
    print(f">>> RESUMING from checkpoint pushed to {HF_REPO_ID}/last-checkpoint "
          f"(local copy: {RESUME_CKPT_DIR})")
else:
    print(f">>> STARTING FRESH -- no last-checkpoint found in {HF_REPO_ID} yet")


In [ ]:
# Model + tokenizer: try Unsloth first, fall back to plain TRL-style PEFT + bitsandbytes
# QLoRA if Unsloth can't be installed cleanly (version conflict) or would touch torch.
use_unsloth = False

try:
    import importlib
    importlib.import_module("unsloth")
    print("unsloth already importable, skipping install")
    unsloth_ready = True
except ImportError:
    unsloth_ready = safe_pip_install(["unsloth"], "unsloth")

if unsloth_ready:
    try:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=torch.float16, load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES, use_gradient_checkpointing=True, random_state=SEED,
            bias="none",
        )
        use_unsloth = True
        print("using Unsloth")
    except Exception as e:
        print(f"unsloth installed but failed to load the model ({e}) -- falling back to TRL + PEFT")

if not use_unsloth:
    for pkg, import_name in [("peft", "peft"), ("bitsandbytes", "bitsandbytes"),
                              ("trl", "trl"), ("accelerate", "accelerate")]:
        try:
            __import__(import_name)
        except ImportError:
            ok = safe_pip_install([pkg], pkg)
            if not ok:
                raise RuntimeError(f"installing {pkg} would have changed the torch build -- "
                                    f"stopping rather than risk a broken environment")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={"": 0},
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("using TRL-style PEFT + bitsandbytes (Unsloth unavailable)")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # training, not generation -- labels align left-to-right
model.config.use_cache = False  # incompatible with gradient checkpointing during training
model.print_trainable_parameters()


In [ ]:
# Load + tokenize the Step 6 training data. Loss is masked to the SQL answer tokens
# only: prompt_text and full_text are tokenized fresh with THIS notebook's tokenizer
# (not trusted from the locally-precomputed token counts), and the length difference is
# exactly how many leading tokens to mask -- see build_training_example in src/prompts.py
# for why prompt_text is guaranteed to be an exact prefix of full_text. That invariant is
# only guaranteed at the *string* level there; here we verify it actually holds at the
# *token* level for this notebook's own tokenizer, for every example, and stop rather
# than silently train with a wrong mask if it doesn't.
import json
import random
from collections import Counter

from datasets import Dataset


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def prefilter_by_length(rows):
    # coarse pre-filter on the precomputed token count (cheap -- skips re-tokenizing the
    # baseball_1 outliers entirely); build_dataset below applies the exact filter
    return [r for r in rows if r["full_tokens"] <= MAX_SEQ_LEN]


def tokenize_row(row):
    prompt_ids = tokenizer(row["prompt_text"], add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(row["full_text"], add_special_tokens=False)["input_ids"]
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    prefix_ok = full_ids[:len(prompt_ids)] == prompt_ids
    return {"input_ids": full_ids, "labels": labels, "length": len(full_ids)}, prefix_ok


def build_dataset(rows, name):
    examples, mismatches = [], []
    for r in rows:
        ex, prefix_ok = tokenize_row(r)
        if len(ex["input_ids"]) > MAX_SEQ_LEN:
            continue
        examples.append(ex)
        if not prefix_ok:
            mismatches.append((r["db_id"], r["question"]))
    if mismatches:
        print(f"MASKING CHECK FAILED on {name}: {len(mismatches)} example(s) where the "
              f"tokenized prompt is not an exact prefix of the tokenized full text:")
        for db_id, q in mismatches[:10]:
            print(f"  [{db_id}] {q}")
        raise RuntimeError(
            f"{len(mismatches)} masking mismatch(es) on {name} -- stopping before training. "
            f"Training would otherwise silently use a wrong loss mask for these examples."
        )
    print(f"masking check passed on {name}: tokenized prompt is an exact prefix of "
          f"tokenized full_text for all {len(examples)} examples")
    return Dataset.from_list(examples)


train_rows_raw = load_jsonl(TRAIN_PATH)
val_rows_raw = load_jsonl(VAL_PATH)

train_rows = prefilter_by_length(train_rows_raw)
val_rows = prefilter_by_length(val_rows_raw)

# Fixed random (seed 42) subset of the validation set, drawn across ALL of its db_ids --
# not the first N, which (Spider's json groups rows by db_id) would have been dominated
# by whichever one or two db_ids happen to come first in file order.
val_sample_n = min(FIXED_VAL_SUBSET_SIZE, len(val_rows))
val_sample_idx = sorted(random.Random(42).sample(range(len(val_rows)), val_sample_n))
val_rows_sampled = [val_rows[i] for i in val_sample_idx]

full_counts = Counter(r["db_id"] for r in val_rows)
sample_counts = Counter(r["db_id"] for r in val_rows_sampled)
print(f"val eval subset: {len(val_rows_sampled)}/{len(val_rows)} examples (seed 42), "
      f"covering {len(sample_counts)}/{len(full_counts)} validation db_ids")
print("  per-db_id counts in the sample:", dict(sample_counts))

train_dataset = build_dataset(train_rows, "train")
val_dataset = build_dataset(val_rows_sampled, "val (fixed eval subset)")

print(f"train: {len(train_dataset)}/{len(train_rows_raw)} kept (<= {MAX_SEQ_LEN} tokens)")
print(f"val (fixed eval subset): {len(val_dataset)} examples")


In [ ]:
# Verify the stop-token handling: full_text should end with the target SQL followed by
# <|im_end|>, and that <|im_end|> must be in the TRAINED (unmasked) region, not masked out
# -- otherwise the model would never learn to actually stop generating.
example = train_dataset[0]
ids, labels = example["input_ids"], example["labels"]
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

boundary = next(i for i, l in enumerate(labels) if l != -100)  # first trained token
print("=== masking example (train_dataset[0]) ===")
print(f"total tokens: {len(ids)}, masked (prompt): {boundary}, trained (target): {len(ids) - boundary}")

print("\n-- tokens around the mask/train boundary --")
for i in range(max(0, boundary - 5), min(len(ids), boundary + 5)):
    tag = "MASK " if labels[i] == -100 else "TRAIN"
    print(f"  [{i}] {tag} {ids[i]:>7d} {tokenizer.decode([ids[i]])!r}")

print("\n-- last 10 tokens (should end in the SQL target, then <|im_end|>) --")
for i in range(max(0, len(ids) - 10), len(ids)):
    tag = "MASK " if labels[i] == -100 else "TRAIN"
    marker = "  <-- <|im_end|>" if ids[i] == im_end_id else ""
    print(f"  [{i}] {tag} {ids[i]:>7d} {tokenizer.decode([ids[i]])!r}{marker}")

has_im_end_trained = any(t == im_end_id and l != -100 for t, l in zip(ids, labels))
print(f"\n<|im_end|> present and included in the loss (not masked): {has_im_end_trained}")
if not has_im_end_trained:
    raise RuntimeError("the target's <|im_end|> is missing or masked out of the training "
                        "loss -- stopping rather than training a model that never learns to stop.")


In [ ]:
# Custom collator: pads input_ids/attention_mask/labels to the batch's own max length
# (group_by_length below keeps similar-length examples together, so this padding waste
# stays small) and tracks real (non-pad) tokens processed, for the smoke-test throughput
# measurement.
import torch

token_counter = {"total": 0}


def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(b["labels"], dtype=torch.long)
    token_counter["total"] += int(attention_mask.sum().item())
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


In [ ]:
# Weights & Biases -- resume into the same run id across Kaggle sessions if we're
# resuming training, so the loss curve doesn't fragment into separate runs.
import wandb

resume_wandb_id = None
if RESUMED:
    id_path = os.path.join(RESUME_CKPT_DIR, "wandb_run_id.txt")
    if os.path.exists(id_path):
        with open(id_path, encoding="utf-8") as f:
            resume_wandb_id = f.read().strip() or None

wandb.login(key=WANDB_API_KEY)
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, id=resume_wandb_id, resume="allow")
print("W&B run id:", wandb.run.id, "(resumed)" if resume_wandb_id else "(new)")


In [ ]:
# Trainer setup.
#
# Why these choices for Run A:
# - learning_rate=2e-4: standard for LoRA, roughly 10x a typical full-fine-tune LR --
#   the adapter starts at zero (LoRA's B matrix is zero-initialized) and only ~0.1-1% of
#   parameters are being updated, so it needs a larger step size to move meaningfully in
#   one epoch.
# - cosine schedule, warmup_steps = ceil(0.03 * total_steps): a short warmup avoids a
#   destabilizing LR spike on the first few steps (when LoRA's output is still close to
#   zero), then a smooth cosine decay suits a single short run better than a step
#   schedule with no natural restart. Computed as an explicit integer step count (not
#   warmup_ratio) because that argument doesn't exist in every transformers version --
#   see the fail-fast check right after the config cell.
# - per_device_batch=1, grad_accum=32 -> effective batch 32: group_by_length means a
#   batch can contain several of the longest (~4,096-token) sequences together, and
#   that's the real memory risk on a T4's 16GB -- never batching more than one such
#   sequence at a time removes it. Accumulation makes up the full effective batch size.
#   build_trainer() below can still shrink this further (and grow accumulation to match)
#   if an actual CUDA OOM happens, as a safety net, not the primary defense.
# - train_sampling_strategy="group_by_length": without it, a batch containing one long
#   outlier forces every other sequence in that batch to pad up to it -- expensive when
#   lengths range from ~250 to ~4,096 tokens (Step 6 stats).
import math
import time

from transformers import Trainer, TrainerCallback, TrainingArguments

effective_batch_size = PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
total_steps = max(1, -(-len(train_dataset) // effective_batch_size)) * NUM_EPOCHS
eval_steps = max(1, total_steps // 4)
warmup_steps = math.ceil(WARMUP_RATIO * total_steps)
print(f"effective batch size: {effective_batch_size}, total optimizer steps: {total_steps}, "
      f"eval every {eval_steps} steps, warmup for {warmup_steps} steps")


def build_training_args(per_device_batch, grad_accum):
    return TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        warmup_steps=warmup_steps,
        fp16=True,
        gradient_checkpointing=True,
        train_sampling_strategy="group_by_length",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="no",  # custom time-based checkpointing via the callback below
        report_to=["wandb"],
        seed=SEED,
        optim="adamw_8bit" if use_unsloth else "paged_adamw_8bit",
        remove_unused_columns=False,
    )


class ThroughputAndCheckpointCallback(TrainerCallback):
    """Reports real tokens/sec + a projected full-epoch time once, after
    SMOKE_TEST_STEPS steps, without stopping training. Also pushes a full resumable
    checkpoint (adapter + optimizer + scheduler + trainer state) to the HF Hub at
    `last-checkpoint` at least every TARGET_CHECKPOINT_INTERVAL_SEC, and once more at
    the very end."""

    def __init__(self, trainer_ref, total_steps):
        self.trainer_ref = trainer_ref
        self.total_steps = total_steps
        self.train_start = None
        self.last_checkpoint_time = None
        self.smoke_test_reported = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        self.last_checkpoint_time = self.train_start

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if not self.smoke_test_reported and state.global_step >= SMOKE_TEST_STEPS:
            elapsed = now - self.train_start
            tok_per_sec = token_counter["total"] / elapsed if elapsed > 0 else 0.0
            sec_per_step = elapsed / state.global_step
            projected_sec = sec_per_step * self.total_steps
            print(f"\n[smoke test] after {state.global_step} steps ({elapsed:.0f}s measured): "
                  f"{tok_per_sec:.0f} tokens/sec, {sec_per_step:.2f}s/step")
            print(f"[smoke test] projected time for all {self.total_steps} steps "
                  f"(this epoch): ~{projected_sec / 60:.1f} min -- continuing automatically\n")
            self.smoke_test_reported = True

        if now - self.last_checkpoint_time >= TARGET_CHECKPOINT_INTERVAL_SEC:
            self._checkpoint(state.global_step)
            self.last_checkpoint_time = now

    def on_train_end(self, args, state, control, **kwargs):
        self._checkpoint(state.global_step, final=True)

    def _checkpoint(self, step, final=False):
        trainer = self.trainer_ref["trainer"]
        print(f"saving checkpoint at step {step}{' (final)' if final else ''}...")
        ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoint-latest")
        os.makedirs(ckpt_dir, exist_ok=True)
        trainer.save_model(ckpt_dir)
        trainer.state.save_to_json(os.path.join(ckpt_dir, "trainer_state.json"))
        try:
            torch.save(trainer.optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
            torch.save(trainer.lr_scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
        except Exception as e:
            print(f"  (optimizer/scheduler state not saved: {e} -- a resume would restart "
                  f"optimizer momentum, but adapter weights and step count still carry over)")
        with open(os.path.join(ckpt_dir, "wandb_run_id.txt"), "w", encoding="utf-8") as f:
            f.write(wandb.run.id if wandb.run else "")
        from huggingface_hub import upload_folder
        upload_folder(repo_id=HF_REPO_ID, folder_path=ckpt_dir, path_in_repo="last-checkpoint",
                      token=HF_TOKEN, commit_message=f"checkpoint at step {step}")
        print(f"  pushed to {HF_REPO_ID}/last-checkpoint")


In [ ]:
import time


def build_trainer(per_device_batch, grad_accum):
    args = build_training_args(per_device_batch, grad_accum)
    t = Trainer(model=model, args=args, train_dataset=train_dataset,
                eval_dataset=val_dataset, data_collator=collate_fn)
    trainer_ref["trainer"] = t
    t.add_callback(ThroughputAndCheckpointCallback(trainer_ref, total_steps))
    return t


trainer_ref = {}
per_device_batch, grad_accum = PER_DEVICE_BATCH, GRAD_ACCUM_STEPS
resume_dir = RESUME_CKPT_DIR if RESUMED else None
oom_retries = 0
MAX_OOM_RETRIES = 4

while True:
    trainer = build_trainer(per_device_batch, grad_accum)
    try:
        if resume_dir is not None:
            trainer.train(resume_from_checkpoint=resume_dir)
        else:
            trainer.train()
        break
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        oom_retries += 1
        if oom_retries > MAX_OOM_RETRIES:
            raise RuntimeError(
                f"CUDA OOM persists after {MAX_OOM_RETRIES} retries (currently "
                f"per_device_batch={per_device_batch}) -- needs a shorter max_seq_len "
                f"or a bigger GPU."
            ) from e

        print(f"\nCUDA OOM at per_device_batch={per_device_batch}, grad_accum={grad_accum} "
              f"(retry {oom_retries}/{MAX_OOM_RETRIES}). Saving an emergency checkpoint "
              f"before retrying.")
        torch.cuda.empty_cache()
        emergency_dir = os.path.join(OUTPUT_DIR, "checkpoint-oom-emergency")
        try:
            os.makedirs(emergency_dir, exist_ok=True)
            trainer.save_model(emergency_dir)
            trainer.state.save_to_json(os.path.join(emergency_dir, "trainer_state.json"))
            resume_dir = emergency_dir
            print(f"  saved emergency checkpoint to {emergency_dir}, resuming training from there")
        except Exception as save_err:
            print(f"  could not save an emergency checkpoint ({save_err}) -- resuming from "
                  f"the last regular checkpoint instead (or fresh, if there wasn't one yet)")

        if per_device_batch > 1:
            per_device_batch = max(1, per_device_batch // 2)
            grad_accum = grad_accum * 2
            print(f"  new config: per_device_batch={per_device_batch}, grad_accum={grad_accum} "
                  f"(effective batch size unchanged: {per_device_batch * grad_accum})\n")
        else:
            print(f"  already at per_device_batch=1 -- retrying at the same batch size "
                  f"after clearing the CUDA cache\n")


In [ ]:
# Final save: push the adapter to the HF repo, save loss curves + run metadata locally.
import json

import matplotlib.pyplot as plt

final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

from huggingface_hub import upload_folder
upload_folder(repo_id=HF_REPO_ID, folder_path=final_dir, path_in_repo="final_adapter",
              token=HF_TOKEN, commit_message="final adapter, run A")
print(f"pushed final adapter to {HF_REPO_ID}/final_adapter")

log_history = trainer.state.log_history
train_points = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
eval_points = [(e["step"], e["eval_loss"]) for e in log_history if "eval_loss" in e]

plt.figure(figsize=(8, 5))
if train_points:
    xs, ys = zip(*train_points)
    plt.plot(xs, ys, label="train loss")
if eval_points:
    xs, ys = zip(*eval_points)
    plt.plot(xs, ys, label="val loss", marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Run A: rank 16, alpha 32, 1 epoch")
plt.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve_run_a.png")
print("saved /kaggle/working/loss_curve_run_a.png")

run_metadata = {
    "experiment": "Exp 4: QLoRA run A",
    "model": MODEL_NAME,
    "backend": "unsloth" if use_unsloth else "trl-style peft + bitsandbytes",
    "prompt_version": "v2",
    "lora": {"rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
              "target_modules": LORA_TARGET_MODULES},
    "max_seq_len": MAX_SEQ_LEN,
    "learning_rate": LEARNING_RATE, "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_ratio": WARMUP_RATIO, "warmup_steps": warmup_steps,
    "per_device_batch_initial": PER_DEVICE_BATCH, "grad_accum_steps_initial": GRAD_ACCUM_STEPS,
    "per_device_batch_final": per_device_batch, "grad_accum_steps_final": grad_accum,
    "oom_retries": oom_retries, "effective_batch_size": effective_batch_size,
    "num_epochs": NUM_EPOCHS, "seed": SEED,
    "n_train_examples": len(train_dataset), "n_eval_examples": len(val_dataset),
    "total_steps": total_steps,
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "gpu_name": torch.cuda.get_device_name(0),
    "hf_repo": HF_REPO_ID, "wandb_run_id": wandb.run.id if wandb.run else None,
    "train_log_history": log_history,
}
with open("/kaggle/working/run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print("saved /kaggle/working/run_metadata.json")
wandb.finish()


## Download from `/kaggle/working/` when done:
- `loss_curve_run_a.png`
- `run_metadata.json`

The adapter itself doesn't need downloading -- it's already pushed to
`HF_REPO_ID/final_adapter` on the Hub, which Step 8's evaluation notebook will load
directly.

If you stop partway through, nothing extra to do: the next `last-checkpoint` push already
happened within the last ~25 minutes, and opening this notebook fresh and running it again
will find it and resume.
